# Studio Proyek 3 — Fine-Tuning Transformer Pretrained untuk Klasifikasi Sentimen SmSA

**Mata Kuliah:** Pembelajaran Mendalam (IF25-40401)  
**Dataset:** SmSA — Sentence-level Sentiment Analysis (IndoNLU, Wilie dkk., 2020)  
**Tugas:** Klasifikasi sentimen 3 kelas: *positive*, *neutral*, *negative*

---

### Rancangan Eksperimen

**Variabel Bebas (Independent Variables):**
1. **Backbone Model** (3 pilihan): `bert-base-multilingual-cased`, `distilbert-base-multilingual-cased`, `indobenchmark/indobert-base-p1`
2. **Strategi Fine-Tuning** (3 pilihan): Full Fine-Tuning, Partial Freezing, Linear Probing
3. **Random Seed** (3 seed): `[42, 123, 2026]`

**Variabel Terkendali (Controlled Variables):** Dikunci identik di semua run.
| Variabel | Nilai |
|---|---|
| Dataset | SmSA (100%, split resmi) |
| Max Sequence Length | 128 |
| Batch Size | 32 (fallback: 16 jika OOM) |
| Epochs (Full & Partial) | 3 |
| Epochs (Linear Probing) | 5 |
| LR (Full & Partial) | 3e-5 |
| LR (Linear Probing) | 1e-3 |
| Optimizer | AdamW (weight_decay=0.01) |
| Warmup Ratio | 0.1 |

**Variabel Terikat (Dependent Variables / Metrik):**  
Accuracy, Macro F1, Training Time (s), Inference Latency (ms/sentence), Trainable Params, Peak VRAM (MB)

**Hipotesis:**
- **H1:** IndoBERT (spesialisasi Bahasa Indonesia) akan menghasilkan F1 Makro tertinggi dibanding BERT & DistilBERT multilingual.
- **H2:** Full Fine-Tuning akan menghasilkan akurasi tertinggi namun membutuhkan waktu latih terlama; Linear Probing akan konvergen paling lambat namun paling hemat VRAM.
- **H3:** DistilBERT akan memiliki latensi inferensi terendah (paling cepat) karena arsitektur yang lebih kecil (6 layer vs 12 layer).

**Total Eksperimen:** 3 model × 3 strategi × 3 seed = **27 run**

---
## Sel 0 — Instalasi Dependensi

Instalasi library yang diperlukan. Jalankan sekali di awal sesi, lalu *restart runtime* jika diminta.

In [ ]:
# Install semua dependensi yang dibutuhkan
#!pip install -q transformers==4.40.2 datasets accelerate scikit-learn pandas matplotlib seaborn

---
## Sel 1 — Import Library & Konfigurasi Eksperimen

Semua hyperparameter dikumpulkan dalam satu dictionary `CFG` sebagai *single source of truth*. Hal ini memastikan konsistensi dan kemudahan pengubahan konfigurasi tanpa harus menelusuri seluruh notebook.

In [14]:
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings("ignore")

# ─── Konfigurasi Eksperimen (Single Source of Truth) ───────────────────────
CFG = {
    # Dataset
    "data_dir"         : "data/smsa",
    "num_labels"       : 3,
    "label2id"         : {"positive": 0, "neutral": 1, "negative": 2},
    "id2label"         : {0: "positive", 1: "neutral", 2: "negative"},

    # Backbone model yang diuji
    "models": {
        "BERT"     : "bert-base-multilingual-cased",
        "DistilBERT": "distilbert-base-multilingual-cased",
        "IndoBERT" : "indobenchmark/indobert-base-p1",
    },

    # Strategi Fine-Tuning
    "strategies"       : ["full", "partial", "probing"],

    # Random Seeds 
    "seeds"            : [42, 123, 2026],

    # Hyperparameter Terkendali
    "max_seq_len"      : 128,
    "batch_size"       : 32,
    "epochs_full"      : 3,
    "epochs_partial"   : 3,
    "epochs_probing"   : 5,
    "lr_full"          : 3e-5,  # Wajib dalam rentang [2e-5, 5e-5]
    "lr_partial"       : 3e-5,  # Wajib dalam rentang [2e-5, 5e-5]
    "lr_probing"       : 1e-3,  # Wajib dalam rentang [1e-3, 5e-3]
    "weight_decay"     : 0.01,
    "warmup_ratio"     : 0.1,

    # Output
    "results_dir"      : "results",
    "raw_csv"          : "hasil_smsa_studio3.csv",
    "agg_csv"          : "rekap_agregat_smsa_studio3.csv",
}

# Deteksi perangkat komputasi
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Perangkat: {DEVICE}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f"Perangkat: {DEVICE} (Apple Silicon GPU)")
    print("Catatan: Peak VRAM tracking tidak tersedia di MPS, akan dilaporkan 0.0 MB.")
else:
    DEVICE = torch.device("cpu")
    print(f"Perangkat: {DEVICE} (tidak ada GPU terdeteksi)")

os.makedirs(CFG["results_dir"], exist_ok=True)
print(f"\nKonfigurasi eksperimen siap. Total run: {len(CFG['models'])} model × {len(CFG['strategies'])} strategi × {len(CFG['seeds'])} seed = {len(CFG['models'])*len(CFG['strategies'])*len(CFG['seeds'])} run")

Perangkat: mps (Apple Silicon GPU)
Catatan: Peak VRAM tracking tidak tersedia di MPS, akan dilaporkan 0.0 MB.

Konfigurasi eksperimen siap. Total run: 3 model × 3 strategi × 3 seed = 27 run


---
## Sel 2 — Pemuatan & Pembersihan Dataset

Dataset SmSA dimuat dari berkas TSV yang sudah diunduh sebelumnya. Sesuai **spesifikasi wajib**, tiga langkah pembersihan dilakukan:
1. **Hapus teks duplikat** — memastikan tidak ada bias dari pengulangan data.
2. **Hapus label contradiction** — teks yang sama dengan label berbeda akan dikecualikan karena ambigu.
3. **Hapus kebocoran data** (*data leakage*) — teks validasi/uji yang bocor ke data latih dihapus dari set latih.

In [15]:
def load_smsa(data_dir: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Memuat dan membersihkan dataset SmSA sesuai spesifikasi."""
    def read_tsv(split: str) -> pd.DataFrame:
        path = os.path.join(data_dir, f"{split}_preprocess.tsv")
        return pd.read_csv(path, sep="\t", header=None, names=["text", "sentiment"], quoting=3)

    df_train = read_tsv("train")
    df_valid = read_tsv("valid")
    df_test  = read_tsv("test")

    n_before = len(df_train)

    # 1. Hapus label contradiction (teks sama, label berbeda)
    contradictory = df_train.groupby("text")["sentiment"].nunique()
    contradictory_texts = set(contradictory[contradictory > 1].index)
    df_train = df_train[~df_train["text"].isin(contradictory_texts)]

    # 2. Hapus duplikat teks (ambil satu, setelah contradiction sudah bersih)
    df_train = df_train.drop_duplicates(subset=["text"]).reset_index(drop=True)

    # 3. Hapus kebocoran data (teks valid/test yang ada di train)
    leak_texts = set(df_valid["text"]) | set(df_test["text"])
    df_train = df_train[~df_train["text"].isin(leak_texts)].reset_index(drop=True)

    print(f"[Train] Sebelum pembersihan: {n_before:,} baris → Setelah: {len(df_train):,} baris")
    print(f"  - Contradiction: {len(contradictory_texts)} teks dihapus")
    print(f"  - Duplikat & Leakage: tersisihkan pada proses drop_duplicates & filter")
    return df_train, df_valid, df_test


df_train, df_valid, df_test = load_smsa(CFG["data_dir"])

# Distribusi kelas
print("\nDistribusi Kelas:")
for split_name, df in [("Train", df_train), ("Valid", df_valid), ("Test", df_test)]:
    counts = df["sentiment"].value_counts().to_dict()
    print(f"  {split_name:5s} ({len(df):,} baris): {counts}")

[Train] Sebelum pembersihan: 11,000 baris → Setelah: 10,919 baris
  - Contradiction: 0 teks dihapus
  - Duplikat & Leakage: tersisihkan pada proses drop_duplicates & filter

Distribusi Kelas:
  Train (10,919 baris): {'positive': 6376, 'negative': 3407, 'neutral': 1136}
  Valid (1,260 baris): {'positive': 735, 'negative': 394, 'neutral': 131}
  Test  (500 baris): {'positive': 208, 'negative': 204, 'neutral': 88}


---
## Sel 3 — PyTorch Dataset & Fungsi Utilitas

`SmSADataset` adalah wrapper PyTorch `Dataset` yang menangani tokenisasi on-the-fly. Desain ini memisahkan tanggung jawab data preparation dari loop pelatihan, mengikuti prinsip *separation of concerns*. Fungsi-fungsi utilitas (seed, parameter counting, VRAM, inferensi) dikumpulkan di sini sebagai *helper functions*.

In [16]:
class SmSADataset(Dataset):
    """PyTorch Dataset untuk dataset SmSA."""

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int, label2id: dict):
        self.texts  = df["text"].tolist()
        self.labels = df["sentiment"].map(label2id).tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }


def set_seed(seed: int) -> None:
    """Set semua sumber randomness untuk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def count_parameters(model) -> tuple[int, int]:
    """Menghitung total dan trainable parameters suatu model."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def get_peak_vram_mb() -> float:
    """Mengembalikan peak VRAM usage dalam MB.
    Hanya tersedia pada CUDA. MPS dan CPU mengembalikan 0.0.
    """
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated(DEVICE) / 1024**2
    return 0.0  # MPS tidak menyediakan API memory tracking publik


def measure_inference_latency(model, tokenizer, texts: list, max_len: int, n_warmup: int = 10) -> float:
    """
    Mengukur rata-rata latensi inferensi (ms/kalimat) secara akurat.
    Warmup run dilakukan terlebih dahulu untuk menghindari overhead CUDA JIT.
    Hasilnya valid HANYA jika dijalankan dalam satu sesi runtime yang sama (spesifikasi).
    """
    model.eval()
    sample = texts[:min(100, len(texts))]

    with torch.no_grad():
        # Warmup
        for text in sample[:n_warmup]:
            enc = tokenizer(text, max_length=max_len, padding="max_length",
                            truncation=True, return_tensors="pt").to(DEVICE)
            _ = model(**enc)

        # Timing
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elif DEVICE.type == "mps":
            torch.mps.synchronize()
        t_start = time.perf_counter()
        for text in sample:
            enc = tokenizer(text, max_length=max_len, padding="max_length",
                            truncation=True, return_tensors="pt").to(DEVICE)
            _ = model(**enc)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elif DEVICE.type == "mps":
            torch.mps.synchronize()
        t_end = time.perf_counter()

    return (t_end - t_start) / len(sample) * 1000  # ms/kalimat


print("Fungsi utilitas berhasil didefinisikan.")

Fungsi utilitas berhasil didefinisikan.


---
## Sel 4 — Fungsi Pembekuan Layer (Freezing)

Implementasi strategi freezing sesuai **spesifikasi wajib**:
- **`full`**: Semua parameter dilatih (tidak ada yang dibekukan).
- **`partial`**: Embedding + separuh awal encoder dibekukan (BERT/IndoBERT: layer 0–5; DistilBERT: layer 0–2).
- **`probing`**: Seluruh backbone dibekukan, hanya classification head yang dilatih.

Urutan N_params yang wajib dibuktikan: **Probing < Partial < Full**.

In [17]:
def apply_freezing(model, strategy: str, model_name: str) -> None:
    """
    Menerapkan strategi pembekuan parameter sesuai spesifikasi.
    Modifikasi dilakukan in-place pada model.
    """
    if strategy == "full":
        # Semua parameter dapat dilatih
        for param in model.parameters():
            param.requires_grad = True

    elif strategy == "probing":
        # Linear Probing: bekukan seluruh backbone, latih hanya head
        for param in model.parameters():
            param.requires_grad = False
        # Aktifkan hanya classifier head
        for param in model.classifier.parameters():
            param.requires_grad = True
        # DistilBERT menggunakan 'pre_classifier' sebelum 'classifier'
        if hasattr(model, "pre_classifier"):
            for param in model.pre_classifier.parameters():
                param.requires_grad = True

    elif strategy == "partial":
        # Partial Freezing: bekukan embedding + separuh awal encoder
        for param in model.parameters():
            param.requires_grad = False  # Bekukan semua dulu

        is_distilbert = "distilbert" in model_name.lower()
        # Tentukan indeks layer yang DILATIH (separuh akhir)
        if is_distilbert:
            # 6 layer total → layer 0-2 beku, layer 3-5 dilatih
            trainable_layer_indices = [3, 4, 5]
            encoder_attr = "transformer.layer"
        else:
            # BERT / IndoBERT: 12 layer total → layer 0-5 beku, layer 6-11 dilatih
            trainable_layer_indices = list(range(6, 12))
            encoder_attr = "encoder.layer"

        # Aktifkan layer encoder bagian akhir
        # Navigasi nested attribute (bert.encoder.layer atau distilbert.transformer.layer)
        base_model = getattr(model, model.base_model_prefix)
        # Pecah encoder_attr (bisa nested seperti 'encoder.layer')
        encoder_layers = base_model
        for attr in encoder_attr.split("."):
            encoder_layers = getattr(encoder_layers, attr)

        for idx in trainable_layer_indices:
            for param in encoder_layers[idx].parameters():
                param.requires_grad = True

        # Aktifkan classifier head
        for param in model.classifier.parameters():
            param.requires_grad = True
        if hasattr(model, "pre_classifier"):
            for param in model.pre_classifier.parameters():
                param.requires_grad = True


print("Fungsi apply_freezing berhasil didefinisikan.")

Fungsi apply_freezing berhasil didefinisikan.


---
## Sel 5 — Fungsi Training & Evaluasi

Dua fungsi inti eksperimen:
- **`train_one_epoch`**: Satu epoch forward + backward pass. Mengembalikan rata-rata loss.
- **`evaluate`**: Evaluasi pada satu DataLoader. Mengembalikan accuracy, macro F1, dan total waktu evaluasi.

In [18]:
def train_one_epoch(model, loader, optimizer, scheduler) -> float:
    """Satu epoch training. Mengembalikan rata-rata loss."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader) -> tuple[float, float]:
    """Evaluasi model pada satu DataLoader. Mengembalikan (accuracy, macro_f1)."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=-1).cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

    acc      = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return acc, macro_f1


print("Fungsi train_one_epoch & evaluate berhasil didefinisikan.")

Fungsi train_one_epoch & evaluate berhasil didefinisikan.


---
## Sel 6 — Fungsi Eksperimen Utama (run_experiment)

Satu fungsi `run_experiment` mengorkestrasikan satu run penuh (satu kombinasi model + strategi + seed). Fungsi ini mengembalikan dictionary hasil yang berisi semua metrik wajib.

In [19]:
def run_experiment(
    model_key: str,
    model_name: str,
    strategy: str,
    seed: int,
    df_train: pd.DataFrame,
    df_valid: pd.DataFrame,
    df_test: pd.DataFrame,
    cfg: dict,
) -> dict:
    """
    Menjalankan satu run eksperimen lengkap.
    Mengembalikan dict berisi seluruh metrik wajib.
    """
    set_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    # Tentukan LR & epoch berdasarkan strategi
    lr     = cfg["lr_probing"]  if strategy == "probing" else (
             cfg["lr_partial"]  if strategy == "partial" else cfg["lr_full"])
    epochs = cfg["epochs_probing"] if strategy == "probing" else (
             cfg["epochs_partial"]  if strategy == "partial" else cfg["epochs_full"])

    # Tokenizer & DataLoader
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def make_loader(df, shuffle=False):
        ds = SmSADataset(df, tokenizer, cfg["max_seq_len"], cfg["label2id"])
        return DataLoader(ds, batch_size=cfg["batch_size"], shuffle=shuffle,
                  num_workers=0,  # 0 wajib di macOS (spawn-based multiprocessing)
                  pin_memory=(DEVICE.type == "cuda"))  # pin_memory hanya valid di CUDA

    train_loader = make_loader(df_train, shuffle=True)
    valid_loader = make_loader(df_valid)
    test_loader  = make_loader(df_test)

    # Model: Head Replacement (num_labels=3 otomatis mengganti head bawaan)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=cfg["num_labels"],
        id2label=cfg["id2label"],
        label2id=cfg["label2id"],
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    # Terapkan strategi freezing
    apply_freezing(model, strategy, model_name)

    total_params, trainable_params = count_parameters(model)

    # Optimizer & Scheduler
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=cfg["weight_decay"]
    )
    total_steps  = len(train_loader) * epochs
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # ─── Training Loop ───────────────────────────────────────────────────────
    t_train_start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_acc, val_f1 = evaluate(model, valid_loader)
        print(f"  [{model_key}/{strategy}/seed={seed}] Epoch {epoch}/{epochs} "
              f"| Loss: {loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    t_train_end = time.perf_counter()
    training_time_s = t_train_end - t_train_start

    # ─── Evaluasi Akhir pada Test Set ────────────────────────────────────────
    test_acc, test_f1 = evaluate(model, test_loader)
    peak_vram_mb      = get_peak_vram_mb()

    # ─── Latensi Inferensi (ms/kalimat) ─────────────────────────────────────
    inference_latency_ms = measure_inference_latency(
        model, tokenizer, df_test["text"].tolist(), cfg["max_seq_len"]
    )

    result = {
        "model"                 : model_key,
        "model_name"            : model_name,
        "strategy"              : strategy,
        "seed"                  : seed,
        "test_accuracy"         : round(test_acc, 4),
        "test_macro_f1"         : round(test_f1, 4),
        "training_time_s"       : round(training_time_s, 2),
        "inference_latency_ms"  : round(inference_latency_ms, 4),
        "total_params"          : total_params,
        "trainable_params"      : trainable_params,
        "peak_vram_mb"          : round(peak_vram_mb, 2),
        "val_accuracy"          : round(val_acc, 4),
        "val_macro_f1"          : round(val_f1, 4),
    }

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
    return result


print("Fungsi run_experiment berhasil didefinisikan.")

Fungsi run_experiment berhasil didefinisikan.


---
## Sel 7 — Verifikasi Hierarki Parameter (Pre-Eksperimen)

Sebelum eksperimen penuh dimulai, **spesifikasi mewajibkan pembuktian kode** bahwa urutan jumlah parameter terlatih memenuhi:

$$N_{params}(\text{Probing}) < N_{params}(\text{Partial}) < N_{params}(\text{Full})$$

Verifikasi dilakukan sekali pada setiap backbone dengan seed default.

In [20]:
print("=" * 70)
print("VERIFIKASI HIERARKI PARAMETER TERLATIH (Spesifikasi Wajib)")
print("=" * 70)
print("Optimasi: setiap backbone dimuat 1x lalu bobot di-reset antar strategi.")
print(f"Estimasi: 3 load model vs 9 load model sebelumnya (~3x lebih cepat).\n")

verification_rows = []

for model_key, model_name in CFG["models"].items():
    print(f"── {model_key} ({model_name}) ──")
    param_counts = {}

    # Muat model SEKALI per backbone
    tmp_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=CFG["num_labels"],
        ignore_mismatched_sizes=True,
    )

    for strategy in CFG["strategies"]:
        # Reset semua requires_grad ke True sebelum menerapkan strategi baru
        for param in tmp_model.parameters():
            param.requires_grad = True

        apply_freezing(tmp_model, strategy, model_name)
        total, trainable = count_parameters(tmp_model)
        param_counts[strategy] = trainable
        print(f"  [{strategy:7s}] Total: {total:,} | Trainable: {trainable:,} "              f"({trainable/total*100:.1f}%)")
        verification_rows.append({
            "model": model_key, "strategy": strategy,
            "total_params": total, "trainable_params": trainable
        })

    del tmp_model  # Bebaskan memori setelah 3 strategi selesai diverifikasi

    # Verifikasi urutan hierarki
    assert param_counts["probing"] < param_counts["partial"] < param_counts["full"], \
        f"GAGAL hierarki parameter untuk {model_key}!"
    print(f"  ✅ VALID: Probing ({param_counts['probing']:,}) < "          f"Partial ({param_counts['partial']:,}) < "          f"Full ({param_counts['full']:,})\n")

print("=" * 70)
print("Semua hierarki parameter VALID. Eksperimen dapat dilanjutkan.")
print("=" * 70)


VERIFIKASI HIERARKI PARAMETER TERLATIH (Spesifikasi Wajib)
Optimasi: setiap backbone dimuat 1x lalu bobot di-reset antar strategi.
Estimasi: 3 load model vs 9 load model sebelumnya (~3x lebih cepat).

── BERT (bert-base-multilingual-cased) ──


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7101.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from t

  [full   ] Total: 177,855,747 | Trainable: 177,855,747 (100.0%)
  [partial] Total: 177,855,747 | Trainable: 42,529,539 (23.9%)
  [probing] Total: 177,855,747 | Trainable: 2,307 (0.0%)
  ✅ VALID: Probing (2,307) < Partial (42,529,539) < Full (177,855,747)

── DistilBERT (distilbert-base-multilingual-cased) ──


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6961.50it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [full   ] Total: 135,326,979 | Trainable: 135,326,979 (100.0%)
  [partial] Total: 135,326,979 | Trainable: 21,856,515 (16.2%)
  [probing] Total: 135,326,979 | Trainable: 592,899 (0.4%)
  ✅ VALID: Probing (592,899) < Partial (21,856,515) < Full (135,326,979)

── IndoBERT (indobenchmark/indobert-base-p1) ──


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 68883.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [full   ] Total: 124,443,651 | Trainable: 124,443,651 (100.0%)
  [partial] Total: 124,443,651 | Trainable: 42,529,539 (34.2%)
  [probing] Total: 124,443,651 | Trainable: 2,307 (0.0%)
  ✅ VALID: Probing (2,307) < Partial (42,529,539) < Full (124,443,651)

Semua hierarki parameter VALID. Eksperimen dapat dilanjutkan.


---
## Sel 8 — Eksperimen Utama (27 Run)

Loop eksperimen mengiterasi semua kombinasi model × strategi × seed. Setiap hasil disimpan secara inkremental ke `hasil_smsa_studio3.csv` agar tidak hilang jika runtime terputus (Colab/Kaggle).

**Estimasi waktu (GPU T4):** ~90–120 menit total.

In [ ]:
all_results = []
raw_csv_path = os.path.join(CFG["results_dir"], CFG["raw_csv"])

total_runs = len(CFG["models"]) * len(CFG["strategies"]) * len(CFG["seeds"])
run_count  = 0

for model_key, model_name in CFG["models"].items():
    for strategy in CFG["strategies"]:
        for seed in CFG["seeds"]:
            run_count += 1
            print(f"\n{'='*65}")
            print(f"Run {run_count}/{total_runs}: {model_key} | {strategy.upper()} | Seed={seed}")
            print(f"{'='*65}")

            result = run_experiment(
                model_key=model_key,
                model_name=model_name,
                strategy=strategy,
                seed=seed,
                df_train=df_train,
                df_valid=df_valid,
                df_test=df_test,
                cfg=CFG,
            )
            all_results.append(result)

            # Simpan inkremental ke CSV (aman dari interrupsi runtime)
            df_results = pd.DataFrame(all_results)
            df_results.to_csv(raw_csv_path, index=False)

            print(f"  → Test Acc: {result['test_accuracy']:.4f} "
                  f"| Test F1: {result['test_macro_f1']:.4f} "
                  f"| Train Time: {result['training_time_s']:.1f}s "
                  f"| Latency: {result['inference_latency_ms']:.2f} ms/sent")

print(f"\n{'='*65}")
print(f"Semua {total_runs} run selesai! Hasil disimpan di: {raw_csv_path}")
print(f"{'='*65}")


Run 1/27: BERT | FULL | Seed=42


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17767.93it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

---
## Sel 9 — Agregasi Hasil (Mean ± Std)

Sesuai **spesifikasi wajib**, hasil akhir dilaporkan sebagai **Rata-rata ± Simpangan Baku** ($\text{Mean} \pm \text{Std}$) dari ketiga seed. Dilarang hanya melaporkan skor tertinggi (*cherry-picking*).

File agregat disimpan sebagai `rekap_agregat_smsa_studio3.csv`.

In [ ]:
df_raw = pd.read_csv(raw_csv_path)

# Metrik yang diagregasi
metric_cols = [
    "test_accuracy", "test_macro_f1",
    "training_time_s", "inference_latency_ms",
    "trainable_params", "peak_vram_mb",
]

# Hitung mean dan std per (model, strategy)
grouped = df_raw.groupby(["model", "strategy"])[metric_cols]
df_mean = grouped.mean().round(4)
df_std  = grouped.std().round(4)

# Gabungkan menjadi format "mean ± std"
df_agg = pd.DataFrame(index=df_mean.index)
for col in metric_cols:
    df_agg[col] = df_mean[col].astype(str) + " ± " + df_std[col].astype(str)

# Tambahkan kolom jumlah seed
df_agg["n_seeds"] = grouped.size().values

# Simpan
agg_csv_path = os.path.join(CFG["results_dir"], CFG["agg_csv"])
df_agg.reset_index().to_csv(agg_csv_path, index=False)

print(f"Hasil agregat disimpan di: {agg_csv_path}")
print("\n=== Rekap Agregat (Mean ± Std dari 3 Seed) ===")
display(df_agg.reset_index())

---
## Sel 10 — Visualisasi Hasil

Empat grafik dibuat untuk mendukung analisis:
1. **Bar plot Test Macro F1** per model dan strategi.
2. **Bar plot Test Accuracy** per model dan strategi.
3. **Scatter plot Training Time vs Accuracy** untuk analisis efisiensi.
4. **Bar plot Inference Latency** per model (perbandingan kecepatan inferensi).

In [ ]:
# Siapkan data mean untuk visualisasi
df_plot = df_mean.reset_index()

strategy_order = ["full", "partial", "probing"]
model_order    = ["BERT", "DistilBERT", "IndoBERT"]
palette        = {"full": "#4C72B0", "partial": "#55A868", "probing": "#C44E52"}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Studio Proyek 3 — Hasil Fine-Tuning SmSA (Mean dari 3 Seed)",
             fontsize=14, fontweight="bold", y=1.01)

# ── Plot 1: Test Macro F1 ──────────────────────────────────────────────────
ax1 = axes[0, 0]
for strat in strategy_order:
    subset = df_plot[df_plot["strategy"] == strat]
    subset = subset.set_index("model").reindex(model_order)
    x = range(len(model_order))
    offset = strategy_order.index(strat) * 0.25 - 0.25
    ax1.bar([i + offset for i in x], subset["test_macro_f1"],
            width=0.25, label=strat.capitalize(), color=palette[strat], alpha=0.85)

ax1.set_title("Test Macro F1 per Model & Strategi")
ax1.set_ylabel("Macro F1")
ax1.set_xticks(range(len(model_order)))
ax1.set_xticklabels(model_order)
ax1.set_ylim(0, 1.05)
ax1.legend(title="Strategi")
ax1.grid(axis="y", alpha=0.3)

# ── Plot 2: Test Accuracy ──────────────────────────────────────────────────
ax2 = axes[0, 1]
for strat in strategy_order:
    subset = df_plot[df_plot["strategy"] == strat]
    subset = subset.set_index("model").reindex(model_order)
    x = range(len(model_order))
    offset = strategy_order.index(strat) * 0.25 - 0.25
    ax2.bar([i + offset for i in x], subset["test_accuracy"],
            width=0.25, label=strat.capitalize(), color=palette[strat], alpha=0.85)

ax2.set_title("Test Accuracy per Model & Strategi")
ax2.set_ylabel("Accuracy")
ax2.set_xticks(range(len(model_order)))
ax2.set_xticklabels(model_order)
ax2.set_ylim(0, 1.05)
ax2.legend(title="Strategi")
ax2.grid(axis="y", alpha=0.3)

# ── Plot 3: Training Time vs Accuracy (Scatter) ────────────────────────────
ax3 = axes[1, 0]
marker_map = {"BERT": "o", "DistilBERT": "s", "IndoBERT": "^"}  
for strat in strategy_order:
    for mdl in model_order:
        row = df_plot[(df_plot["strategy"] == strat) & (df_plot["model"] == mdl)]
        if len(row) > 0:
            ax3.scatter(row["training_time_s"].values[0], row["test_accuracy"].values[0],
                        color=palette[strat], marker=marker_map[mdl], s=120,
                        label=f"{mdl}/{strat}" if strat == "full" else "_nolegend_",
                        zorder=5)
            ax3.annotate(f"{mdl[:4]}/{strat[:3]}",
                         (row["training_time_s"].values[0], row["test_accuracy"].values[0]),
                         textcoords="offset points", xytext=(5, 3), fontsize=7)

ax3.set_title("Trade-off: Training Time vs Test Accuracy")
ax3.set_xlabel("Training Time (s)")
ax3.set_ylabel("Test Accuracy")
ax3.grid(alpha=0.3)

# ── Plot 4: Inference Latency per Model ────────────────────────────────────
ax4 = axes[1, 1]
latency_by_model = df_plot.groupby("model")["inference_latency_ms"].mean().reindex(model_order)
bars = ax4.bar(model_order, latency_by_model.values, color=["#4C72B0", "#55A868", "#C44E52"],
               alpha=0.85, edgecolor="white")
for bar, val in zip(bars, latency_by_model.values):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f"{val:.2f} ms", ha="center", va="bottom", fontsize=9)

ax4.set_title("Rata-rata Inference Latency per Model (ms/kalimat)")
ax4.set_ylabel("Latency (ms/kalimat)")
ax4.grid(axis="y", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CFG["results_dir"], "visualisasi_hasil.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafik disimpan di: {plot_path}")

---
## Sel 11 — Analisis & Pembahasan

Bagian ini menjawab **3 Pertanyaan Riset Wajib** sesuai spesifikasi.

In [ ]:
df_raw = pd.read_csv(raw_csv_path)

# ── H1: Apakah IndoBERT menghasilkan F1 tertinggi? ─────────────────────────
print("\n📊 H1 — Perbandingan Rata-rata Test Macro F1 per Model (Semua Strategi):")
h1 = df_raw.groupby("model")["test_macro_f1"].agg(["mean", "std"]).round(4)
h1.columns = ["Mean F1", "Std F1"]
print(h1.reindex(["BERT", "DistilBERT", "IndoBERT"]))

# ── H2: Apakah Full > Partial > Probing dalam akurasi? ─────────────────────
print("\n📊 H2 — Rata-rata Test Accuracy per Strategi (Semua Model):")
h2 = df_raw.groupby("strategy")["test_accuracy"].agg(["mean", "std"]).round(4)
h2.columns = ["Mean Acc", "Std Acc"]
print(h2.reindex(["full", "partial", "probing"]))

# ── H3: Apakah DistilBERT paling cepat? ────────────────────────────────────
print("\n📊 H3 — Rata-rata Inference Latency per Model (ms/kalimat):")
h3 = df_raw.groupby("model")["inference_latency_ms"].agg(["mean", "std"]).round(4)
h3.columns = ["Mean Latency (ms)", "Std Latency (ms)"]
print(h3.reindex(["BERT", "DistilBERT", "IndoBERT"]))

# ── Tabel Rekap Lengkap ─────────────────────────────────────────────────────
print("\n📊 Rekap Lengkap (Mean ± Std):")
df_agg_display = df_agg.reset_index()[["model", "strategy", "test_accuracy", "test_macro_f1",
                                        "training_time_s", "inference_latency_ms", "trainable_params", "peak_vram_mb"]]
display(df_agg_display)

---
## Sel 12 — Confusion Matrix Model Terbaik

Confusion matrix ditampilkan untuk kombinasi model dan strategi dengan **Test Macro F1 tertinggi** (best run), sebagai analisis kesalahan klasifikasi per kelas sentimen.

In [ ]:
from sklearn.metrics import confusion_matrix

# Temukan kombinasi terbaik berdasarkan rata-rata Test Macro F1
best_row = df_mean["test_macro_f1"].idxmax()  # (model, strategy)
best_model_key, best_strategy = best_row
best_seed = df_raw[(df_raw["model"] == best_model_key) &
                   (df_raw["strategy"] == best_strategy)]["seed"].iloc[0]

print(f"Kombinasi Terbaik: {best_model_key} | {best_strategy.upper()} | Seed={best_seed}")
print(f"Mean Test Macro F1: {df_mean.loc[best_row, 'test_macro_f1']:.4f}")

# Re-load model terbaik untuk confusion matrix
set_seed(best_seed)
best_model_name = CFG["models"][best_model_key]
tokenizer       = AutoTokenizer.from_pretrained(best_model_name)
model_best      = AutoModelForSequenceClassification.from_pretrained(
    best_model_name, num_labels=CFG["num_labels"],
    ignore_mismatched_sizes=True
).to(DEVICE)
apply_freezing(model_best, best_strategy, best_model_name)

# Re-train model terbaik
train_loader_best = DataLoader(
    SmSADataset(df_train, tokenizer, CFG["max_seq_len"], CFG["label2id"]),
    batch_size=CFG["batch_size"], shuffle=True, num_workers=0
)
optimizer_best = AdamW(
    filter(lambda p: p.requires_grad, model_best.parameters()),
    lr=CFG["lr_full"], weight_decay=CFG["weight_decay"]
)
epochs_best = CFG["epochs_probing"] if best_strategy == "probing" else CFG["epochs_full"]
total_steps_best = len(train_loader_best) * epochs_best
scheduler_best = get_linear_schedule_with_warmup(
    optimizer_best, int(total_steps_best * CFG["warmup_ratio"]), total_steps_best
)
for _ in range(epochs_best):
    train_one_epoch(model_best, train_loader_best, optimizer_best, scheduler_best)

# Inferensi pada test set
model_best.eval()
all_preds_best, all_labels_best = [], []
test_loader_best = DataLoader(
    SmSADataset(df_test, tokenizer, CFG["max_seq_len"], CFG["label2id"]),
    batch_size=CFG["batch_size"], num_workers=0
)
with torch.no_grad():
    for batch in test_loader_best:
        out = model_best(input_ids=batch["input_ids"].to(DEVICE),
                         attention_mask=batch["attention_mask"].to(DEVICE))
        all_preds_best.extend(out.logits.argmax(-1).cpu().tolist())
        all_labels_best.extend(batch["labels"].tolist())

# Plot Confusion Matrix
class_names = ["positive", "neutral", "negative"]
cm = confusion_matrix(all_labels_best, all_preds_best)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title(f"Confusion Matrix — {best_model_key} / {best_strategy.upper()} (Seed={best_seed})")
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(os.path.join(CFG["results_dir"], "confusion_matrix_best.png"), dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels_best, all_preds_best, target_names=class_names))

del model_best

---
## Sel 13 — Ringkasan & Validasi Output

Memastikan semua berkas luaran wajib telah dihasilkan dan siap dikumpulkan.

In [ ]:
print("\n" + "=" * 65)
print("VALIDASI BERKAS LUARAN")
print("=" * 65)

output_files = {
    "Raw Run CSV"    : raw_csv_path,
    "Agregat CSV"    : agg_csv_path,
    "Visualisasi"    : os.path.join(CFG["results_dir"], "visualisasi_hasil.png"),
    "Confusion Matrix": os.path.join(CFG["results_dir"], "confusion_matrix_best.png"),
}

all_ok = True
for label, fpath in output_files.items():
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1024 if exists else 0
    status = f"✅ ({size:.1f} KB)" if exists else "❌ TIDAK ADA"
    print(f"  {label:20s}: {fpath}  →  {status}")
    all_ok = all_ok and exists

print()
if all_ok:
    print("✅ Semua berkas luaran berhasil dihasilkan.")
    print("   Langkah selanjutnya:")
    print("   1. File → Print → Save as PDF (untuk PDF notebook)")
    print("   2. Upload ke Google Drive / Kaggle dan salin tautan aktif")
    print("   3. Lampirkan kedua file CSV ke form pengumpulan")
else:
    print("❌ Ada berkas yang belum dihasilkan. Periksa ulang log error di atas.")

# Ringkasan performa eksperimen
print("\n" + "=" * 65)
print("RINGKASAN EKSPERIMEN")
print("=" * 65)
print(f"  Total run           : {len(df_raw)} run ({len(CFG['models'])} model × {len(CFG['strategies'])} strategi × {len(CFG['seeds'])} seed)")
print(f"  Best Model          : {df_raw.loc[df_raw['test_macro_f1'].idxmax(), 'model']}")
print(f"  Best Strategy       : {df_raw.loc[df_raw['test_macro_f1'].idxmax(), 'strategy'].upper()}")
print(f"  Best Test Macro F1  : {df_raw['test_macro_f1'].max():.4f}")
print(f"  Best Test Accuracy  : {df_raw.loc[df_raw['test_macro_f1'].idxmax(), 'test_accuracy']:.4f}")
print(f"  Fastest Inference   : {df_raw.groupby('model')['inference_latency_ms'].mean().idxmin()} "
      f"({df_raw.groupby('model')['inference_latency_ms'].mean().min():.2f} ms/sent)")
print("=" * 65)